# 📄 FIR Document Quality Analysis & Zone-wise Visualization
---

This notebook performs:

### ✅ 1. Recursive PDF Quality Analysis
### ✅ 2. Zone-wise Document Score Aggregation
### ✅ 3. Graph Generation
### - Document Score Charts
### - Page Distribution Boxplots
### - Heatmaps of Page Quality
### - Zone Summary Bar Chart

Output report:
- `fir_quality_report.json`
- Plots in `output_graphs/`


In [1]:
import os
import fitz  # PyMuPDF
import cv2
import numpy as np
from pdf2image import convert_from_path
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

print("Libraries loaded.")

Libraries loaded.


# 📌 Quality Metric Functions
These functions evaluate:
- Sharpness
- Contrast
- Noise
- Skew angle
- Composite quality score

In [2]:
def compute_sharpness(img_gray):
    return cv2.Laplacian(img_gray, cv2.CV_64F).var()

def compute_contrast(img_gray):
    return img_gray.std()

def compute_noise(img_gray):
    mean = np.mean(img_gray)
    std = np.std(img_gray)
    return std / mean if mean != 0 else 0

def compute_skew(img_gray):
    coords = np.column_stack(np.where(img_gray < 200))
    if len(coords) == 0:
        return 0
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    return abs(angle)

def score_page(sharp, contrast, noise, skew):
    sharp_score = min(sharp / 100, 1)
    contrast_score = min(contrast / 80, 1)
    noise_score = min(noise / 0.1, 1)
    skew_score = max(0, 1 - (skew / 10))

    return round(0.35 * sharp_score +
                 0.30 * contrast_score +
                 0.20 * noise_score +
                 0.15 * skew_score, 3)

print("Quality functions ready.")

Quality functions ready.


# 📌 PDF Analysis Function
Converts PDF → images → calculates metrics for each page.

In [8]:
def analyze_pdf(pdf_path):
    print("Analyzing:", pdf_path)
    poppler_path = "/opt/homebrew/bin" 
    pages = convert_from_path(pdf_path, dpi=200,poppler_path=poppler_path)
    print("pages",pages)
    results = []

    for page_num, page in enumerate(pages, start=1):
        img = np.array(page)
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        sharp = compute_sharpness(img_gray)
        contrast = compute_contrast(img_gray)
        noise = compute_noise(img_gray)
        skew = compute_skew(img_gray)
        score = score_page(sharp, contrast, noise, skew)

        results.append({
            "page": page_num,
            "sharpness": sharp,
            "contrast": contrast,
            "noise": noise,
            "skew_angle": skew,
            "quality_score": score
        })

    pdf_score = round(sum(r["quality_score"] for r in results) / len(results), 3)
    return results, pdf_score

# 📌 Zone-wise Recursive Processing
Walks through CZ / NZ / WZ / SZ directories and builds a full JSON report.

In [11]:
def process_all_zones(root_folder="/Users/vidhya/Projects/cb-cid/data/FIR"):
    zones = ["CZ", "NZ", "WZ", "SZ"]
    # zones = [ "SZ"]
    final_output = {}

    for zone in zones:
        zone_path = os.path.join(root_folder, zone)
        if not os.path.isdir(zone_path):
            print(f"   Skipping {zone}, directory not found.")
            continue

        print(f"\nProcessing Zone: {zone}")
        zone_result = {}
        zone_scores = []

        for root, dirs, files in os.walk(zone_path):
            for file in files:
                if file.lower().endswith(".pdf"):
                    pdf_path = os.path.join(root, file)
                    print(f"  - Analyzing: {pdf_path}")

                    try:
                        page_results, pdf_score = analyze_pdf(pdf_path)
                        zone_result[file] = {
                            "path": pdf_path,
                            "document_score": pdf_score,
                            "pages": page_results
                        }
                        zone_scores.append(pdf_score)

                    except Exception as e:
                        zone_result[file] = {"error": str(e)}

        zone_summary_score = round(sum(zone_scores)/len(zone_scores), 3) if zone_scores else 0

        final_output[zone] = {
            "zone_average_score": zone_summary_score,
            "documents": zone_result
        }

    return final_output

print("Zone processor ready.")

Zone processor ready.


# 📌 Run Quality Pipeline & Save JSON Report

In [12]:
report = process_all_zones()

with open("fir_quality_report.json", "w") as f:
    json.dump(report, f, indent=4)

print("\n✔️ FIR Quality Report Saved → fir_quality_report.json")


Processing Zone: CZ
  - Analyzing: /Users/vidhya/Projects/cb-cid/data/FIR/CZ/Trichy CBCID Cr.No.02 of 2022 Suspetious drawing death.pdf
Analyzing: /Users/vidhya/Projects/cb-cid/data/FIR/CZ/Trichy CBCID Cr.No.02 of 2022 Suspetious drawing death.pdf
pages [<PIL.PpmImagePlugin.PpmImageFile image mode=RGB size=1654x2339 at 0x17EB54910>, <PIL.PpmImagePlugin.PpmImageFile image mode=RGB size=1654x2339 at 0x17EB56DD0>, <PIL.PpmImagePlugin.PpmImageFile image mode=RGB size=1654x2339 at 0x17EB54650>, <PIL.PpmImagePlugin.PpmImageFile image mode=RGB size=1654x2339 at 0x17EB54A10>]
  - Analyzing: /Users/vidhya/Projects/cb-cid/data/FIR/CZ/Karur Velayutham palayam PS.Cr.No.339 of 2011  Theft case  UN.pdf
Analyzing: /Users/vidhya/Projects/cb-cid/data/FIR/CZ/Karur Velayutham palayam PS.Cr.No.339 of 2011  Theft case  UN.pdf
pages [<PIL.PpmImagePlugin.PpmImageFile image mode=RGB size=1653x2339 at 0x17EB54310>, <PIL.PpmImagePlugin.PpmImageFile image mode=RGB size=1653x2339 at 0x17EB54910>]
  - Analyzing: 

# 📊 Graph Generation Section
Creates:
- Document Score Bar Charts
- Page Score Boxplots
- Heatmaps
- Zone Summary Chart

In [ ]:
os.makedirs("output_graphs", exist_ok=True)

with open("fir_quality_report.json", "r") as f:
    data = json.load(f)

# -------------------------------------------
# 1️⃣ Document Scores – Bar Graph with Labels
# -------------------------------------------
def plot_zone_document_scores(zone_name, zone_data):
    docs = zone_data["documents"]
    names, scores = [], []

    for doc, info in docs.items():
        if "document_score" in info:
            names.append(doc)
            scores.append(info["document_score"])

    if not scores:
        return

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=names, y=scores, palette="viridis")

    # Add value labels on top of bars
    for i, v in enumerate(scores):
        ax.text(i, v + 0.02, f"{v:.2f}", 
                ha='center', fontsize=10, fontweight="bold")

    plt.xticks(rotation=45, ha="right")
    plt.title(f"Document Scores – {zone_name}", fontsize=14)
    plt.ylim(0, 1.1)
    plt.tight_layout()
    plt.savefig(f"/Users/vidhya/Projects/cb-cid/output_graphs/{zone_name}_document_scores.png")
    plt.close()

# --------------------------------------------------
# 2️⃣ Page Score Boxplot with Mean Marker + Label
# --------------------------------------------------
def plot_zone_page_boxplot(zone_name, zone_data):
    pages = []
    for doc, info in zone_data["documents"].items():
        if "pages" in info:
            for p in info["pages"]:
                pages.append(p["quality_score"])

    if not pages:
        return

    plt.figure(figsize=(6, 6))
    sns.boxplot(y=pages, color="skyblue")

    # Add mean line
    mean_val = sum(pages) / len(pages)
    plt.axhline(mean_val, color='red', linestyle='--', linewidth=1.2)
    plt.text(0, mean_val + 0.02, f"Mean: {mean_val:.2f}", 
             color='red', fontsize=10, fontweight="bold")

    plt.title(f"Page Score Distribution – {zone_name}", fontsize=14)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(f"/Users/vidhya/Projects/cb-cid/output_graphs/{zone_name}_page_boxplot.png")
    plt.close()

# -----------------------------------------
# 3️⃣ Heatmap with Annotated Page Scores
# -----------------------------------------
def plot_zone_heatmap(zone_name, zone_data):
    rows = []
    docs = zone_data["documents"]

    for doc, info in docs.items():
        if "pages" in info:
            for p in info["pages"]:
                rows.append({
                    "document": doc,
                    "page": p["page"],
                    "score": p["quality_score"]
                })

    if not rows:
        return

    df = pd.DataFrame(rows)
    pivot = df.pivot(index="document", columns="page", values="score")

    plt.figure(figsize=(12, 6))
    sns.heatmap(pivot, annot=True, cmap="YlGnBu", vmin=0, vmax=1,
                fmt=".2f", annot_kws={"size": 9, "weight": "bold"})

    plt.title(f"Page Heatmap – {zone_name}", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"/Users/vidhya/Projects/cb-cid/output_graphs/{zone_name}_heatmap.png")
    plt.close()

# ------------------------------------------------
# 4️⃣ Zone Summary (Bar Plot with Value Labels)
# ------------------------------------------------
def plot_zone_summary(data):
    zones = []
    scores = []

    for zone, info in data.items():
        if isinstance(info.get("zone_average_score"), (float, int)):
            zones.append(zone)
            scores.append(info["zone_average_score"])

    plt.figure(figsize=(8, 6))
    ax = sns.barplot(x=zones, y=scores, palette="coolwarm")

    # Add value labels
    for i, v in enumerate(scores):
        ax.text(i, v + 0.02, f"{v:.2f}", 
                ha='center', fontsize=11, fontweight="bold")

    plt.title("Zone Average Quality Scores", fontsize=14)
    plt.ylim(0, 1.1)
    plt.tight_layout()
    plt.savefig("/Users/vidhya/Projects/cb-cid/output_graphs/zone_summary.png")
    plt.close()


# 📈 Generate All Graphs

In [14]:
for zone_name, zone_data in data.items():
    print(f"Generating graphs for {zone_name}...")
    plot_zone_document_scores(zone_name, zone_data)
    plot_zone_page_boxplot(zone_name, zone_data)
    plot_zone_heatmap(zone_name, zone_data)

plot_zone_summary(data)
print("\n✔️ All graphs saved in output_graphs/")

Generating graphs for CZ...


/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/1595748563.py:19: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=names, y=scores, palette="viridis")


Generating graphs for NZ...
Generating graphs for WZ...


/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/1595748563.py:19: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=names, y=scores, palette="viridis")
/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/1595748563.py:19: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=names, y=scores, palette="viridis")
/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/1595748563.py:23: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Generating graphs for SZ...

✔️ All graphs saved in output_graphs/


/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/1595748563.py:19: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=names, y=scores, palette="viridis")
/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/1595748563.py:79: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=zones, y=scores, palette="coolwarm")


In [19]:
os.makedirs("output_graphs", exist_ok=True)

with open("fir_quality_report.json", "r") as f:
    data = json.load(f)

# -------------------------------------------
# 1️⃣ Document Scores – Bar Graph with Labels
# -------------------------------------------
def plot_zone_document_scores(zone_name, zone_data):
    docs = zone_data["documents"]
    names, scores = [], []

    for doc, info in docs.items():
        if "document_score" in info:
            names.append(doc)
            scores.append(info["document_score"])

    if not scores:
        return

    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=names, y=scores, palette="viridis")

    # Add value labels on top of bars
    for i, v in enumerate(scores):
        ax.text(i, v + 0.02, f"{v:.2f}", 
                ha='center', fontsize=10, fontweight="bold")

    plt.xticks(rotation=45, ha="right")
    plt.title(f"Document Scores – {zone_name}", fontsize=14)
    plt.ylim(0, 1.1)
    plt.tight_layout()
    plt.savefig(f"/Users/vidhya/Projects/cb-cid/output_graphs/{zone_name}_document_scores.png")
    plt.close()

# --------------------------------------------------
# 2️⃣ Page Score Boxplot with Mean Marker + Label
# --------------------------------------------------
def plot_zone_page_boxplot(zone_name, zone_data):
    pages = []
    for doc, info in zone_data["documents"].items():
        if "pages" in info:
            for p in info["pages"]:
                pages.append(p["quality_score"])

    if not pages:
        return

    plt.figure(figsize=(6, 6))
    sns.boxplot(y=pages, color="skyblue")

    # Add mean line
    mean_val = sum(pages) / len(pages)
    plt.axhline(mean_val, color='red', linestyle='--', linewidth=1.2)
    plt.text(0, mean_val + 0.02, f"Mean: {mean_val:.2f}", 
             color='red', fontsize=10, fontweight="bold")

    plt.title(f"Page Score Distribution – {zone_name}", fontsize=14)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(f"/Users/vidhya/Projects/cb-cid/output_graphs/{zone_name}_page_boxplot.png")
    plt.close()

# -----------------------------------------
# 3️⃣ Heatmap with Annotated Page Scores
# -----------------------------------------
def plot_zone_heatmap(zone_name, zone_data):
    rows = []
    docs = zone_data["documents"]

    for doc, info in docs.items():
        if "pages" in info:
            for p in info["pages"]:
                rows.append({
                    "document": doc,
                    "page": p["page"],
                    "score": p["quality_score"]
                })

    if not rows:
        return

    df = pd.DataFrame(rows)
    pivot = df.pivot(index="document", columns="page", values="score")

    plt.figure(figsize=(12, 6))
    sns.heatmap(pivot, annot=True, cmap="YlGnBu", vmin=0, vmax=1,
                fmt=".2f", annot_kws={"size": 9, "weight": "bold"})

    plt.title(f"Page Heatmap – {zone_name}", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"/Users/vidhya/Projects/cb-cid/output_graphs/{zone_name}_heatmap.png")
    plt.close()

# ------------------------------------------------
# 4️⃣ Zone Summary (Bar Plot with Value Labels)
# ------------------------------------------------
def plot_zone_summary(data):
    zones = []
    scores = []

    for zone, info in data.items():
        if isinstance(info.get("zone_average_score"), (float, int)):
            zones.append(zone)
            scores.append(info["zone_average_score"])

    plt.figure(figsize=(8, 6))
    ax = sns.barplot(x=zones, y=scores, palette="coolwarm")

    # Add value labels
    for i, v in enumerate(scores):
        ax.text(i, v + 0.02, f"{v:.2f}", 
                ha='center', fontsize=11, fontweight="bold")

    plt.title("Zone Average Quality Scores", fontsize=14)
    plt.ylim(0, 1.1)
    plt.tight_layout()
    plt.savefig("/Users/vidhya/Projects/cb-cid/output_graphs/zone_summary.png")
    plt.close()


In [17]:
for zone_name, zone_data in data.items():
    print(f"Generating graphs for {zone_name}...")
    plot_zone_document_scores(zone_name, zone_data)
    plot_zone_page_boxplot(zone_name, zone_data)
    plot_zone_heatmap(zone_name, zone_data)

plot_zone_summary(data)
print("\n✔️ All graphs saved in output_graphs/")

Generating graphs for CZ...
Generating graphs for NZ...


/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3744291965.py:22: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=names, y=scores, palette="viridis")
/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3744291965.py:22: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=names, y=scores, palette="viridis")


Generating graphs for WZ...


/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3744291965.py:22: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=names, y=scores, palette="viridis")
/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3744291965.py:32: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Generating graphs for SZ...


/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3744291965.py:22: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=names, y=scores, palette="viridis")
/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3744291965.py:108: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=zones, y=scores, palette="coolwarm")



✔️ All graphs saved in output_graphs/


In [21]:
# ================================================
# 🔍 Zone Comparison Charts
# ================================================

# Prepare comparison data
zone_comparison = {
    "zone": [],
    "avg_score": [],
    "min_score": [],
    "max_score": [],
    "doc_count": [],
    "page_count": []
}

print("\n📊 Processing zones for comparison...")
for zone, info in data.items():
    docs = info["documents"]
    all_scores = []
    total_pages = 0
    error_count = 0
    
    print(f"\n  {zone}:")
    for doc, doc_info in docs.items():
        if "document_score" in doc_info:
            all_scores.append(doc_info["document_score"])
            if "pages" in doc_info:
                total_pages += len(doc_info["pages"])
        elif "error" in doc_info:
            error_count += 1
    
    print(f"    ✓ Valid documents: {len(all_scores)}")
    print(f"    ✗ Failed documents (with errors): {error_count}")
    print(f"    Total pages: {total_pages}")
    
    if all_scores:
        zone_comparison["zone"].append(zone)
        zone_comparison["avg_score"].append(info["zone_average_score"])
        zone_comparison["min_score"].append(min(all_scores))
        zone_comparison["max_score"].append(max(all_scores))
        zone_comparison["doc_count"].append(len(all_scores))
        zone_comparison["page_count"].append(total_pages)

comparison_df = pd.DataFrame(zone_comparison)

# 1️⃣ Zone Comparison - Min/Max/Average Scores
plt.figure(figsize=(10, 6))
x = range(len(comparison_df))
width = 0.25

bars1 = plt.bar([i - width for i in x], comparison_df["min_score"], width, label="Min Score", color="lightcoral", alpha=0.8)
bars2 = plt.bar([i for i in x], comparison_df["avg_score"], width, label="Avg Score", color="skyblue", alpha=0.8)
bars3 = plt.bar([i + width for i in x], comparison_df["max_score"], width, label="Max Score", color="lightgreen", alpha=0.8)

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.xlabel("Zone", fontsize=12, fontweight='bold')
plt.ylabel("Quality Score", fontsize=12, fontweight='bold')
plt.title("Zone Comparison - Min/Average/Max Quality Scores", fontsize=14, fontweight='bold')
plt.xticks(x, comparison_df["zone"])
plt.legend()
plt.ylim(0, 1.1)
plt.tight_layout()
plt.savefig("/Users/vidhya/Projects/cb-cid/output_graphs/zone_comparison_scores.png", dpi=300)
plt.close()

# 2️⃣ Zone Comparison - Document Count
plt.figure(figsize=(8, 6))
bars = plt.bar(comparison_df["zone"], comparison_df["doc_count"], color="steelblue", alpha=0.8)

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.1,
            f'{int(height)}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.xlabel("Zone", fontsize=12, fontweight='bold')
plt.ylabel("Number of Documents", fontsize=12, fontweight='bold')
plt.title("Zone Comparison - Document Count", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("/Users/vidhya/Projects/cb-cid/output_graphs/zone_comparison_doc_count.png", dpi=300)
plt.close()

# 3️⃣ Zone Comparison - Page Count
plt.figure(figsize=(8, 6))
bars = plt.bar(comparison_df["zone"], comparison_df["page_count"], color="darkorange", alpha=0.8)

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{int(height)}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.xlabel("Zone", fontsize=12, fontweight='bold')
plt.ylabel("Number of Pages", fontsize=12, fontweight='bold')
plt.title("Zone Comparison - Total Page Count", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("/Users/vidhya/Projects/cb-cid/output_graphs/zone_comparison_page_count.png", dpi=300)
plt.close()

# 4️⃣ Zone Comparison - Radar Chart (Multi-metric)
from math import pi

categories = ["Avg Score", "Normalized Doc Count", "Normalized Page Count"]
angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
angles += angles[:1]

plt.figure(figsize=(10, 10))
ax = plt.subplot(111, projection='polar')

# Normalize metrics for radar chart
max_docs = comparison_df["doc_count"].max()
max_pages = comparison_df["page_count"].max()

for idx, zone in enumerate(comparison_df["zone"]):
    values = [
        comparison_df.loc[idx, "avg_score"],
        comparison_df.loc[idx, "doc_count"] / max_docs,
        comparison_df.loc[idx, "page_count"] / max_pages
    ]
    values += values[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2, label=zone)
    ax.fill(angles, values, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_title("Zone Comparison - Multi-Metric Radar Chart", fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True)

plt.tight_layout()
plt.savefig("/Users/vidhya/Projects/cb-cid/output_graphs/zone_comparison_radar.png", dpi=300, bbox_inches='tight')
plt.close()

print("\n✔️ Zone comparison charts generated:")
print("   - zone_comparison_scores.png (Min/Avg/Max comparison)")
print("   - zone_comparison_doc_count.png (Document count)")
print("   - zone_comparison_page_count.png (Page count)")
print("   - zone_comparison_radar.png (Multi-metric radar chart)")
print("\n📊 Comparison Summary:")
print(comparison_df.to_string(index=False))



📊 Processing zones for comparison...

  CZ:
    ✓ Valid documents: 5
    ✗ Failed documents (with errors): 0
    Total pages: 18

  NZ:
    ✓ Valid documents: 7
    ✗ Failed documents (with errors): 0
    Total pages: 23

  WZ:
    ✓ Valid documents: 42
    ✗ Failed documents (with errors): 0
    Total pages: 141

  SZ:
    ✓ Valid documents: 2
    ✗ Failed documents (with errors): 0
    Total pages: 5

✔️ Zone comparison charts generated:
   - zone_comparison_scores.png (Min/Avg/Max comparison)
   - zone_comparison_doc_count.png (Document count)
   - zone_comparison_page_count.png (Page count)
   - zone_comparison_radar.png (Multi-metric radar chart)

📊 Comparison Summary:
zone  avg_score  min_score  max_score  doc_count  page_count
  CZ      0.801      0.751      0.842          5          18
  NZ      0.789      0.681      0.874          7          23
  WZ      0.781      0.692      0.867         42         141
  SZ      0.800      0.786      0.814          2           5

✔️ Zone co

# 📋 Detailed Quality Metrics Summary Report
Aggregates all page-level metrics (sharpness, skew_angle, noise, quality_score) across zones and generates comparative visualizations.

In [24]:
# =====================================================
# 📊 DETAILED QUALITY METRICS SUMMARY REPORT
# =====================================================

print("\n" + "="*60)
print("COMPREHENSIVE QUALITY METRICS ANALYSIS")
print("="*60)

# Load data
with open("/Users/vidhya/Projects/cb-cid/data_quality_graphs/fir_quality_report.json", "r") as f:
    report = json.load(f)

# Initialize metrics aggregator
quality_metrics = {
    "zone": [],
    "avg_sharpness": [],
    "avg_contrast": [],
    "avg_noise": [],
    "avg_skew": [],
    "avg_quality_score": [],
    "page_count": [],
    "doc_count": []
}

# Process each zone
for zone, zone_data in report.items():
    print(f"\n🔍 Processing Zone: {zone}")
    print("-" * 60)
    
    sharpness_values = []
    contrast_values = []
    noise_values = []
    skew_values = []
    quality_scores = []
    total_pages = 0
    doc_count = 0
    
    # Extract metrics from all pages across all documents in zone
    for doc, doc_info in zone_data["documents"].items():
        if "pages" in doc_info:
            doc_count += 1
            for page in doc_info["pages"]:
                total_pages += 1
                sharpness_values.append(page.get("sharpness", 0))
                contrast_values.append(page.get("contrast", 0))
                noise_values.append(page.get("noise", 0))
                skew_values.append(page.get("skew_angle", 0))
                quality_scores.append(page.get("quality_score", 0))
    
    # Calculate averages
    if sharpness_values:
        avg_sharpness = np.mean(sharpness_values)
        avg_contrast = np.mean(contrast_values)
        avg_noise = np.mean(noise_values)
        avg_skew = np.mean(skew_values)
        avg_quality = np.mean(quality_scores)
        
        quality_metrics["zone"].append(zone)
        quality_metrics["avg_sharpness"].append(avg_sharpness)
        quality_metrics["avg_contrast"].append(avg_contrast)
        quality_metrics["avg_noise"].append(avg_noise)
        quality_metrics["avg_skew"].append(avg_skew)
        quality_metrics["avg_quality_score"].append(avg_quality)
        quality_metrics["page_count"].append(total_pages)
        quality_metrics["doc_count"].append(doc_count)
        
        # Print zone summary
        print(f"  📄 Documents: {doc_count}")
        print(f"  📄 Total Pages: {total_pages}")
        print(f"  ✨ Avg Sharpness: {avg_sharpness:.2f}")
        print(f"  📊 Avg Contrast: {avg_contrast:.2f}")
        print(f"  🎯 Avg Noise: {avg_noise:.4f}")
        print(f"  🔄 Avg Skew Angle: {avg_skew:.2f}°")
        print(f"  ⭐ Avg Quality Score: {avg_quality:.3f}")

# Create DataFrame from aggregated metrics
metrics_df = pd.DataFrame(quality_metrics)

print("\n" + "="*60)
print("📊 QUALITY METRICS SUMMARY TABLE")
print("="*60)
print(metrics_df.to_string(index=False))
print("="*60)

# Save summary to CSV
metrics_df.to_csv("/Users/vidhya/Projects/cb-cid/data_quality_graphs/quality_metrics_summary.csv", index=False)
print("\n✔️ Summary saved to: quality_metrics_summary.csv")



COMPREHENSIVE QUALITY METRICS ANALYSIS

🔍 Processing Zone: CZ
------------------------------------------------------------
  📄 Documents: 5
  📄 Total Pages: 18
  ✨ Avg Sharpness: 4059.12
  📊 Avg Contrast: 54.89
  🎯 Avg Noise: 0.2346
  🔄 Avg Skew Angle: 55.35°
  ⭐ Avg Quality Score: 0.809

🔍 Processing Zone: NZ
------------------------------------------------------------
  📄 Documents: 7
  📄 Total Pages: 23
  ✨ Avg Sharpness: 2713.35
  📊 Avg Contrast: 51.50
  🎯 Avg Noise: 0.2183
  🔄 Avg Skew Angle: 58.91°
  ⭐ Avg Quality Score: 0.790

🔍 Processing Zone: WZ
------------------------------------------------------------
  📄 Documents: 42
  📄 Total Pages: 141
  ✨ Avg Sharpness: 2973.86
  📊 Avg Contrast: 49.80
  🎯 Avg Noise: 0.2111
  🔄 Avg Skew Angle: 60.06°
  ⭐ Avg Quality Score: 0.784

🔍 Processing Zone: SZ
------------------------------------------------------------
  📄 Documents: 2
  📄 Total Pages: 5
  ✨ Avg Sharpness: 4774.24
  📊 Avg Contrast: 51.84
  🎯 Avg Noise: 0.2160
  🔄 Avg Skew An

# 📈 Visualization: Quality Metrics Across Zones
Multiple visualization styles to analyze sharpness, contrast, noise, skew, and overall quality scores.

In [28]:
# =====================================================
# 🎨 VISUALIZATION: QUALITY METRICS ACROSS ZONES
# =====================================================

print("\n📈 Generating comprehensive quality metric visualizations...")
print("-" * 60)

# 1️⃣ GROUPED BAR CHART - All Metrics Comparison
print("\n1️⃣ Creating grouped bar chart (All metrics)...")

fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(metrics_df))
width = 0.15

# Normalize metrics to 0-1 scale for fair comparison
sharpness_norm = metrics_df["avg_sharpness"] / metrics_df["avg_sharpness"].max()
contrast_norm = metrics_df["avg_contrast"] / metrics_df["avg_contrast"].max()
noise_norm = metrics_df["avg_noise"] / metrics_df["avg_noise"].max()
skew_norm = metrics_df["avg_skew"] / metrics_df["avg_skew"].max()

bars1 = ax.bar(x - 1.5*width, sharpness_norm, width, label="Sharpness (Normalized)", color="#FF6B6B", alpha=0.85)
bars2 = ax.bar(x - 0.5*width, contrast_norm, width, label="Contrast (Normalized)", color="#4ECDC4", alpha=0.85)
bars3 = ax.bar(x + 0.5*width, noise_norm, width, label="Noise (Normalized)", color="#FFE66D", alpha=0.85)
bars4 = ax.bar(x + 1.5*width, skew_norm, width, label="Skew Angle (Normalized)", color="#95E1D3", alpha=0.85)

# Add value labels
for bars in [bars1, bars2, bars3, bars4]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
               f'{height:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xlabel("Zone", fontsize=12, fontweight='bold')
ax.set_ylabel("Normalized Score", fontsize=12, fontweight='bold')
ax.set_title("Quality Metrics Comparison Across All Zones (Normalized)", fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_df["zone"])
ax.legend(loc='upper right', fontsize=10)
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("/Users/vidhya/Projects/cb-cid/data_quality_graphs/quality_metrics_grouped_bars.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✔️ Saved: quality_metrics_grouped_bars.png")

# 2️⃣ HEATMAP - All Metrics per Zone
print("\n2️⃣ Creating heatmap (All metrics)...")

# Prepare data for heatmap
heatmap_data = metrics_df[[
    "zone", "avg_sharpness", "avg_contrast", "avg_noise", "avg_skew", "avg_quality_score"
]].set_index("zone")

# Normalize for better visualization
heatmap_normalized = heatmap_data.copy()
for col in heatmap_normalized.columns:
    max_val = heatmap_normalized[col].max()
    if max_val > 0:
        heatmap_normalized[col] = heatmap_normalized[col] / max_val

plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_normalized.T, annot=True, fmt=".3f", cmap="RdYlGn", 
            cbar_kws={'label': 'Normalized Score'}, linewidths=0.5, linecolor='gray',
            annot_kws={"fontsize": 10, "fontweight": "bold"})

plt.title("Quality Metrics Heatmap - Zone Comparison (Normalized)", fontsize=14, fontweight='bold', pad=20)
plt.xlabel("Zone", fontsize=12, fontweight='bold')
plt.ylabel("Metric", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("/Users/vidhya/Projects/cb-cid/data_quality_graphs/quality_metrics_heatmap.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✔️ Saved: quality_metrics_heatmap.png")

# 3️⃣ LINE PLOT - Trends Across Zones
print("\n3️⃣ Creating line plot (Metrics trends)...")

plt.figure(figsize=(12, 7))

plt.plot(metrics_df["zone"], sharpness_norm, marker='o', linewidth=2.5, markersize=8, label='Sharpness (Norm)', color='#FF6B6B')
plt.plot(metrics_df["zone"], contrast_norm, marker='s', linewidth=2.5, markersize=8, label='Contrast (Norm)', color='#4ECDC4')
plt.plot(metrics_df["zone"], noise_norm, marker='^', linewidth=2.5, markersize=8, label='Noise (Norm)', color='#FFE66D')
plt.plot(metrics_df["zone"], skew_norm, marker='d', linewidth=2.5, markersize=8, label='Skew Angle (Norm)', color='#95E1D3')

plt.xlabel("Zone", fontsize=12, fontweight='bold')
plt.ylabel("Normalized Score", fontsize=12, fontweight='bold')
plt.title("Quality Metrics Trends Across Zones", fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.1)
plt.tight_layout()
plt.savefig("/Users/vidhya/Projects/cb-cid/data_quality_graphs/quality_metrics_line_plot.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✔️ Saved: quality_metrics_line_plot.png")

# 4️⃣ QUALITY SCORE COMPARISON - Primary Metric
print("\n4️⃣ Creating quality score comparison...")

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#FF6B6B', '#4ECDC4', '#FFE66D', '#95E1D3'][:len(metrics_df)]
bars = ax.bar(metrics_df["zone"], metrics_df["avg_quality_score"], color=colors, alpha=0.85, edgecolor='black', linewidth=1.5)

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
           f'{height:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel("Zone", fontsize=12, fontweight='bold')
ax.set_ylabel("Average Quality Score", fontsize=12, fontweight='bold')
ax.set_title("Average Quality Score by Zone", fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("/Users/vidhya/Projects/cb-cid/data_quality_graphs/quality_score_comparison.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✔️ Saved: quality_score_comparison.png")

# 5️⃣ BOX PLOT - Distribution of Sharpness Across Zones
print("\n5️⃣ Creating box plots (Metric distributions)...")

# Gather individual page-level metrics
page_metrics_data = {"zone": [], "sharpness": [], "contrast": [], "noise": [], "skew": [], "quality_score": []}

for zone, zone_data in report.items():
    for doc, doc_info in zone_data["documents"].items():
        if "pages" in doc_info:
            for page in doc_info["pages"]:
                page_metrics_data["zone"].append(zone)
                page_metrics_data["sharpness"].append(page.get("sharpness", 0))
                page_metrics_data["contrast"].append(page.get("contrast", 0))
                page_metrics_data["noise"].append(page.get("noise", 0))
                page_metrics_data["skew"].append(page.get("skew_angle", 0))
                page_metrics_data["quality_score"].append(page.get("quality_score", 0))

page_metrics_df = pd.DataFrame(page_metrics_data)

# Create subplots for distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sharpness distribution
sns.boxplot(data=page_metrics_df, x="zone", y="sharpness", ax=axes[0, 0], palette="Set2")
axes[0, 0].set_title("Sharpness Distribution by Zone", fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel("Sharpness Value")

# Contrast distribution
sns.boxplot(data=page_metrics_df, x="zone", y="contrast", ax=axes[0, 1], palette="Set2")
axes[0, 1].set_title("Contrast Distribution by Zone", fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel("Contrast Value")

# Noise distribution
sns.boxplot(data=page_metrics_df, x="zone", y="noise", ax=axes[1, 0], palette="Set2")
axes[1, 0].set_title("Noise Distribution by Zone", fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel("Noise Value")

# Skew distribution
sns.boxplot(data=page_metrics_df, x="zone", y="skew", ax=axes[1, 1], palette="Set2")
axes[1, 1].set_title("Skew Angle Distribution by Zone", fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel("Skew Angle (degrees)")

plt.suptitle("Quality Metrics Distribution Across All Zones", fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig("/Users/vidhya/Projects/cb-cid/data_quality_graphs/quality_metrics_distributions.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✔️ Saved: quality_metrics_distributions.png")

# 6️⃣ VIOLIN PLOT - Quality Score Distribution
print("\n6️⃣ Creating violin plot (Quality score distribution)...")

plt.figure(figsize=(12, 7))
sns.violinplot(data=page_metrics_df, x="zone", y="quality_score", palette="muted")
sns.stripplot(data=page_metrics_df, x="zone", y="quality_score", color='black', alpha=0.3, size=3)

plt.xlabel("Zone", fontsize=12, fontweight='bold')
plt.ylabel("Quality Score", fontsize=12, fontweight='bold')
plt.title("Quality Score Distribution Density by Zone", fontsize=14, fontweight='bold')
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("/Users/vidhya/Projects/cb-cid/data_quality_graphs/quality_score_violin_plot.png", dpi=300, bbox_inches='tight')
plt.close()
print("   ✔️ Saved: quality_score_violin_plot.png")

print("\n" + "="*60)
print("✔️ ALL VISUALIZATIONS GENERATED SUCCESSFULLY!")
print("="*60)
print("\n📊 Generated Graphs:")
print("   1. quality_metrics_grouped_bars.png - Normalized metric comparison")
print("   2. quality_metrics_heatmap.png - Heatmap of all metrics")
print("   3. quality_metrics_line_plot.png - Metric trends across zones")
print("   4. quality_score_comparison.png - Primary quality score comparison")
print("   5. quality_metrics_distributions.png - Box plots of metric distributions")
print("   6. quality_score_violin_plot.png - Density distribution of quality scores")
print("="*60)



📈 Generating comprehensive quality metric visualizations...
------------------------------------------------------------

1️⃣ Creating grouped bar chart (All metrics)...
   ✔️ Saved: quality_metrics_grouped_bars.png

2️⃣ Creating heatmap (All metrics)...
   ✔️ Saved: quality_metrics_heatmap.png

3️⃣ Creating line plot (Metrics trends)...
   ✔️ Saved: quality_metrics_line_plot.png

4️⃣ Creating quality score comparison...
   ✔️ Saved: quality_score_comparison.png

5️⃣ Creating box plots (Metric distributions)...
   ✔️ Saved: quality_metrics_heatmap.png

3️⃣ Creating line plot (Metrics trends)...
   ✔️ Saved: quality_metrics_line_plot.png

4️⃣ Creating quality score comparison...
   ✔️ Saved: quality_score_comparison.png

5️⃣ Creating box plots (Metric distributions)...


/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3940315475.py:145: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=page_metrics_df, x="zone", y="sharpness", ax=axes[0, 0], palette="Set2")
/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3940315475.py:150: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=page_metrics_df, x="zone", y="contrast", ax=axes[0, 1], palette="Set2")
/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3940315475.py:155: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(da

   ✔️ Saved: quality_metrics_distributions.png

6️⃣ Creating violin plot (Quality score distribution)...
   ✔️ Saved: quality_score_violin_plot.png

✔️ ALL VISUALIZATIONS GENERATED SUCCESSFULLY!

📊 Generated Graphs:
   1. quality_metrics_grouped_bars.png - Normalized metric comparison
   2. quality_metrics_heatmap.png - Heatmap of all metrics
   3. quality_metrics_line_plot.png - Metric trends across zones
   4. quality_score_comparison.png - Primary quality score comparison
   5. quality_metrics_distributions.png - Box plots of metric distributions
   6. quality_score_violin_plot.png - Density distribution of quality scores


/var/folders/hd/3fjhzq2d2pn72_84n18d57qr0000gn/T/ipykernel_67078/3940315475.py:174: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=page_metrics_df, x="zone", y="quality_score", palette="muted")


# 📊 Statistical Summary Report
Detailed statistics for each metric including mean, median, std, min, max across all zones.

In [31]:
# =====================================================
# 📊 DETAILED STATISTICAL SUMMARY REPORT
# =====================================================

print("\n" + "="*80)
print("DETAILED STATISTICAL ANALYSIS - ALL QUALITY METRICS")
print("="*80)

# Define metrics for analysis
metrics_to_analyze = ["sharpness", "contrast", "noise", "skew", "quality_score"]

# Create comprehensive statistics
stats_summary = {}

for metric in metrics_to_analyze:
    print(f"\n{'='*80}")
    print(f"📊 METRIC: {metric.upper()}")
    print(f"{'='*80}")
    
    metric_data = page_metrics_df[metric]
    
    # Overall statistics
    overall_stats = {
        "Metric": metric,
        "Count": len(metric_data),
        "Mean": metric_data.mean(),
        "Median": metric_data.median(),
        "Std Dev": metric_data.std(),
        "Min": metric_data.min(),
        "25% Quartile": metric_data.quantile(0.25),
        "75% Quartile": metric_data.quantile(0.75),
        "Max": metric_data.max(),
        "Range": metric_data.max() - metric_data.min(),
        "Skewness": page_metrics_df[metric].skew(),
        "Kurtosis": page_metrics_df[metric].kurtosis()
    }
    
    stats_summary[metric] = overall_stats
    
    print(f"\n  🔢 OVERALL STATISTICS:")
    print(f"     Count:          {overall_stats['Count']}")
    print(f"     Mean:           {overall_stats['Mean']:.4f}")
    print(f"     Median:         {overall_stats['Median']:.4f}")
    print(f"     Std Dev:        {overall_stats['Std Dev']:.4f}")
    print(f"     Min:            {overall_stats['Min']:.4f}")
    print(f"     25% Quartile:   {overall_stats['25% Quartile']:.4f}")
    print(f"     75% Quartile:   {overall_stats['75% Quartile']:.4f}")
    print(f"     Max:            {overall_stats['Max']:.4f}")
    print(f"     Range:          {overall_stats['Range']:.4f}")
    print(f"     Skewness:       {overall_stats['Skewness']:.4f}")
    print(f"     Kurtosis:       {overall_stats['Kurtosis']:.4f}")
    
    # Zone-wise statistics
    print(f"\n  📈 ZONE-WISE STATISTICS:")
    print(f"     {'-'*76}")
    print(f"     {'Zone':<8} {'Mean':<12} {'Median':<12} {'Std Dev':<12} {'Min':<12} {'Max':<12}")
    print(f"     {'-'*76}")
    
    for zone in page_metrics_df["zone"].unique():
        zone_data = page_metrics_df[page_metrics_df["zone"] == zone][metric]
        print(f"     {zone:<8} {zone_data.mean():<12.4f} {zone_data.median():<12.4f} {zone_data.std():<12.4f} {zone_data.min():<12.4f} {zone_data.max():<12.4f}")

# Create comprehensive statistics DataFrame
print("\n" + "="*80)
print("📋 COMPREHENSIVE STATISTICS TABLE")
print("="*80)

stats_df = pd.DataFrame(stats_summary).T
print(stats_df.to_string())

# Save statistics to CSV
stats_df.to_csv("/Users/vidhya/Projects/cb-cid/data_quality_graphs/quality_metrics_statistics.csv")
print("\n✔️ Statistics saved to: quality_metrics_statistics.csv")

# Create zone-wise detailed summary
print("\n" + "="*80)
print("📊 ZONE-WISE DETAILED SUMMARY")
print("="*80)

zone_stats_data = []

for zone in metrics_df["zone"]:
    zone_page_data = page_metrics_df[page_metrics_df["zone"] == zone]
    
    zone_stats_data.append({
        "Zone": zone,
        "Docs": metrics_df[metrics_df["zone"] == zone]["doc_count"].values[0],
        "Pages": len(zone_page_data),
        "Sharp_Mean": zone_page_data["sharpness"].mean(),
        "Sharp_Std": zone_page_data["sharpness"].std(),
        "Contrast_Mean": zone_page_data["contrast"].mean(),
        "Contrast_Std": zone_page_data["contrast"].std(),
        "Noise_Mean": zone_page_data["noise"].mean(),
        "Noise_Std": zone_page_data["noise"].std(),
        "Skew_Mean": zone_page_data["skew"].mean(),
        "Skew_Std": zone_page_data["skew"].std(),
        "Quality_Mean": zone_page_data["quality_score"].mean(),
        "Quality_Std": zone_page_data["quality_score"].std(),
    })

zone_stats_df = pd.DataFrame(zone_stats_data)
print(zone_stats_df.to_string(index=False))

# Save zone statistics
zone_stats_df.to_csv("/Users/vidhya/Projects/cb-cid/data_quality_graphs/zone_detailed_statistics.csv", index=False)
print("\n✔️ Zone statistics saved to: zone_detailed_statistics.csv")

print("\n" + "="*80)
print("✔️ STATISTICAL ANALYSIS COMPLETE!")
print("="*80)
print("\n📁 Generated Reports:")
print("   • quality_metrics_statistics.csv - Overall metric statistics")
print("   • zone_detailed_statistics.csv - Zone-wise detailed breakdown")
print("   • quality_metrics_summary.csv - Summary aggregated metrics")
print("="*80)



DETAILED STATISTICAL ANALYSIS - ALL QUALITY METRICS

📊 METRIC: SHARPNESS

  🔢 OVERALL STATISTICS:
     Count:          187
     Mean:           3094.4239
     Median:         2889.7361
     Std Dev:        2520.3042
     Min:            161.5286
     25% Quartile:   1244.0469
     75% Quartile:   4057.0608
     Max:            25320.9161
     Range:          25159.3876
     Skewness:       4.2220
     Kurtosis:       32.8738

  📈 ZONE-WISE STATISTICS:
     ----------------------------------------------------------------------------
     Zone     Mean         Median       Std Dev      Min          Max         
     ----------------------------------------------------------------------------
     CZ       4059.1166    1888.5058    6060.1397    451.0142     25320.9161  
     NZ       2713.3484    2973.6645    1618.9531    291.8788     5166.5018   
     WZ       2973.8649    2889.7361    1806.3159    161.5286     9780.8405   
     SZ       4774.2406    3947.4851    1267.7635    3737.9834 